# Unbekannte Wege

Visualize your Google Maps Timeline, find the streets you've used most and generate a suggested route through the ones you haven't.

## Export your Timeline data

**Android:** Google Maps → profile picture → Settings → Location → Location services → Timeline → *Export Timeline data* → save as `Timeline.json`

**iOS:** Google Maps → profile picture → Settings → Personal content → *Export Timeline data* → save via Files app

## Settings

Set your preferences below before running the notebook.

In [ ]:
networktype = "drive" #Choose "walk" for more accurate routes when you have a lot of walking data, but "drive" is faster to compute and works well enough.
city = "Zürich, Switzerland"  # Change to your city
stepsize = 20 # How many points to skip when plotting the route? Increase this if you have a lot of points to speed up the plotting. Set to 1 to plot all points.
FILENAME = "Timeline.json"  # change to your file

In [ ]:
import json, re, pandas as pd

with open(FILENAME) as f:
    data = json.load(f)

def parse_geo(s):
    nums = re.findall(r"[-\d.]+", str(s).replace("geo:", ""))
    return (float(nums[0]), float(nums[1])) if len(nums) == 2 else None

points = []

# ── Google format: {"semanticSegments": [...]} ─────────────────────────────────
if isinstance(data, dict) and "semanticSegments" in data:
    print("Detected: Google format")
    for seg in data["semanticSegments"]:
        # Rich path points
        for pt in seg.get("timelinePath", []):
            coords = parse_geo(pt["point"])
            if coords:
                points.append({"lat": coords[0], "lon": coords[1], "time": pt["time"]})
        # Activity start/end
        act = seg.get("activity", {})
        for field, tkey in [("start", "startTime"), ("end", "endTime")]:
            raw = act.get(field, {})
            geo = raw.get("latLng", "") if isinstance(raw, dict) else raw
            coords = parse_geo(geo)
            if coords:
                points.append({"lat": coords[0], "lon": coords[1], "time": seg.get(tkey)})
        # Visit location
        vis = seg.get("visit", {})
        geo = vis.get("topCandidate", {}).get("placeLocation", {})
        if isinstance(geo, dict):
            geo = geo.get("latLng", "")
        coords = parse_geo(geo)
        if coords:
            points.append({"lat": coords[0], "lon": coords[1], "time": seg.get("startTime")})

# ── Apple format: [{...}, {...}] ───────────────────────────────────────────────
elif isinstance(data, list):
    print("Detected: Apple format")
    for seg in data:
        start_time = seg.get("startTime")

        # ── timelinePath with offset minutes ──────────────────────────────
        for pt in seg.get("timelinePath", []):
            coords = parse_geo(pt.get("point", ""))
            if not coords:
                continue
            # Compute absolute time from offset if present
            offset = pt.get("durationMinutesOffsetFromStartTime")
            if offset is not None and start_time is not None:
                t = pd.Timestamp(start_time, tz="UTC") + pd.Timedelta(minutes=float(offset))
            else:
                t = start_time
            points.append({"lat": coords[0], "lon": coords[1], "time": t})

        # ── activity start/end ─────────────────────────────────────────────
        act = seg.get("activity", {})
        for field, tkey in [("start", "startTime"), ("end", "endTime")]:
            coords = parse_geo(act.get(field, ""))
            if coords:
                points.append({"lat": coords[0], "lon": coords[1], "time": seg.get(tkey)})

        # ── visit location ─────────────────────────────────────────────────
        vis = seg.get("visit", {})
        geo = vis.get("topCandidate", {}).get("placeLocation", "")
        coords = parse_geo(geo)
        if coords:
            points.append({"lat": coords[0], "lon": coords[1], "time": start_time})

else:
    print("❌ Unrecognised format — please share the top-level structure.")

df_all = pd.DataFrame(points).drop_duplicates()
df_all["time"] = pd.to_datetime(df_all["time"], utc=True)
print(f"\n✓ Extracted {len(df_all):,} points")
print(f"  Date range: {df_all['time'].min().date()} → {df_all['time'].max().date()}")
df_all.head()
df = df_all

In [ ]:
# ── Geocode via Nominatim API directly ────────────────────────────────────────
import requests
resp = requests.get(
    "https://nominatim.openstreetmap.org/search",
    params={"q": city, "format": "json", "limit": 1},
    headers={"User-Agent": "timeline_explorer"}
).json()

bb = resp[0]["boundingbox"]           # [lat_min, lat_max, lon_min, lon_max]
LAT_MIN, LAT_MAX = float(bb[0]), float(bb[1])
LON_MIN, LON_MAX = float(bb[2]), float(bb[3])
df_city = df[
    (df.lat > LAT_MIN) & (df.lat < LAT_MAX) &
    (df.lon > LON_MIN) & (df.lon < LON_MAX)
].copy().reset_index(drop=True)

print(f"Points in city: {len(df_city):,}")
df_city.head()

## All data points in the city

In [ ]:
# ── Map city points + export to PDF via Selenium ──────────────────────────────
import requests, folium, base64, time, os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

print(f"Lat  : {LAT_MIN:.4f} → {LAT_MAX:.4f}")
print(f"Lon  : {LON_MIN:.4f} → {LON_MAX:.4f}")

# ── 2. Filter dataframe ───────────────────────────────────────────────────────
df_city = df[
    (df.lat > LAT_MIN) & (df.lat < LAT_MAX) &
    (df.lon > LON_MIN) & (df.lon < LON_MAX)
].copy().reset_index(drop=True)

print(f"\n✓ Points in city: {len(df_city):,}")

# ── 3. Build folium map fitted to the city bbox ───────────────────────────────
center = ((LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2)
m = folium.Map(location=center, zoom_start=13)

# Fit map view exactly to the city bounding box
m.fit_bounds([[LAT_MIN, LON_MIN], [LAT_MAX, LON_MAX]])

sample = df_city.sample(min(len(df_city), 5_000), random_state=42)
for _, row in sample.iterrows():
    folium.CircleMarker(
        location=(row["lat"], row["lon"]),
        radius=3,
        color="#e74c3c",
        fill=True,
        fill_opacity=0.6,
        weight=1,
    ).add_to(m)

# ── 4. Save HTML + export to PDF ─────────────────────────────────────────────
city_slug = city.split(",")[0].strip().lower().replace(" ", "_")
html_path = f"{city_slug}_map.html"
pdf_path  = f"{city_slug}_map.pdf"

m.save(html_path)
display(m)

def save_map_as_pdf(html_path, pdf_path, wait=5):
    html_abs = "file://" + os.path.abspath(html_path)
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1400,900")
    options.add_experimental_option("prefs", {
        "printing.print_preview_sticky_settings.appState":
            '{"recentDestinations":[{"id":"Save as PDF","origin":"local"}],'
            '"selectedDestinationId":"Save as PDF","version":2}',
    })
    options.add_argument("--kiosk-print-pdf")

    driver = webdriver.Chrome(options=options)
    driver.get(html_abs)
    time.sleep(wait)   # wait for map tiles to load

    pdf_data = driver.execute_cdp_cmd("Page.printToPDF", {
        "paperWidth":    11.69,   # A4 landscape
        "paperHeight":    8.27,
        "printBackground": True,
        "marginTop":    0,
        "marginBottom": 0,
        "marginLeft":   0,
        "marginRight":  0,
    })

    with open(pdf_path, "wb") as f:
        f.write(base64.b64decode(pdf_data["data"]))

    driver.quit()
    print(f"✓ Saved: {pdf_path}")

save_map_as_pdf(html_path, pdf_path, wait=5)

## Build the street network

In [ ]:
import osmnx as ox
CITY = city  # Change to your city in the configuration section at the top
G = ox.graph_from_place(CITY, network_type=networktype) #
print(f"Graph: {len(G.nodes):,} nodes, {len(G.edges):,} edges")

In [ ]:
from tqdm.notebook import tqdm
df_sample = df_city.iloc[::stepsize].reset_index(drop=True)
print(f"Snapping {len(df_sample):,} points to streets...")

edge_counts = {}
for _, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    try:
        edge = ox.nearest_edges(G, row.lon, row.lat)
        edge_counts[edge] = edge_counts.get(edge, 0) + 1
    except:
        pass

print(f"Unique street segments used: {len(edge_counts):,}")
import pickle
with open("edge_counts.pkl", "wb") as f:
    pickle.dump(edge_counts, f)

print("Saved!")

## Resume from checkpoint

The snapping step above can take several minutes. Run this cell instead to reload from the saved pickle.

In [ ]:
import pickle, osmnx as ox

# Reload the street graph
G = ox.graph_from_place(city, network_type=networktype) #make sure you reload the same graph you saved above

# Reload edge counts
with open("edge_counts.pkl", "rb") as f: # reload the same pickle saved above
    edge_counts = pickle.load(f)

print(f"Loaded {len(edge_counts):,} edge counts")

## Choose a starting location

Drag the marker on the map to set your route's starting point.

In [ ]:
from ipyleaflet import Map as LeafMap, Marker, TileLayer, Rectangle
import ipywidgets as widgets
from IPython.display import display

center_lat = (LAT_MIN + LAT_MAX) / 2
center_lon = (LON_MIN + LON_MAX) / 2

pick_map = LeafMap(center=(center_lat, center_lon), zoom=13,
                   layout=widgets.Layout(height="400px"))
pick_map.add_layer(TileLayer(
    url="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attribution="CartoDB"
))

# ── Shaded valid area ──────────────────────────────────────────────────────────
bbox_rect = Rectangle(
    bounds=((LAT_MIN, LON_MIN), (LAT_MAX, LON_MAX)),
    color="#ff00b3",
    fill_color="#ff00b3",
    fill_opacity=0.08,
    weight=1,
    dash_array="6"
)
pick_map.add_layer(bbox_rect)

start_marker = Marker(location=(center_lat, center_lon), draggable=True)
pick_map.add_layer(start_marker)

coord_label = widgets.HTML(
    f"<span style='font-family:monospace;font-size:13px;color:#555;'>"
    f"START_LAT, START_LON = {center_lat:.5f}, {center_lon:.5f}</span>"
)

def update_label(change):
    lat, lon = change["new"]
    coord_label.value = (
        f"<span style='font-family:monospace;font-size:13px;color:#555;'>"
        f"START_LAT, START_LON = {lat:.5f}, {lon:.5f}</span>"
    )

def on_click(**kwargs):
    if kwargs.get("type") == "click":
        lat, lon = kwargs["coordinates"]
        start_marker.location = (lat, lon)

start_marker.observe(update_label, names="location")
pick_map.on_interaction(on_click)

display(widgets.VBox([pick_map, coord_label]))

In [ ]:
START_LAT, START_LON = start_marker.location
print(f"Using start: {START_LAT:.5f}, {START_LON:.5f}")

In [ ]:
import random

def least_used_route(G, edge_counts, start_node, steps=200):
    path = [start_node]
    current = start_node
    prev = None  # track where we came from

    for _ in range(steps):
        neighbors = list(G.successors(current))
        if not neighbors:
            neighbors = list(G.neighbors(current))
        if not neighbors:
            break

        # Score by usage, but heavily penalize going back to previous node
        scores = {}
        for nbr in neighbors:
            count = edge_counts.get((current, nbr, 0), 0)
            if nbr == prev:
                count += 10_000  # big penalty for backtracking
            scores[nbr] = count

        min_score = min(scores.values())
        candidates = [n for n, s in scores.items() if s == min_score]
        next_node = random.choice(candidates)

        prev = current
        path.append(next_node)
        current = next_node

    return path


start_node = ox.nearest_nodes(G, START_LON, START_LAT)
path = least_used_route(G, edge_counts, start_node, steps=200)
print(f"Route has {len(path)} nodes")

In [ ]:
def least_used_route(G, edge_counts, start_node, steps=200, tabu_length=20):
    path = [start_node]
    current = start_node
    visited_edges = set()
    tabu = []  # recently visited nodes

    for _ in range(steps):
        neighbors = list(G.successors(current))
        if not neighbors:
            neighbors = list(G.neighbors(current))
        if not neighbors:
            break

        scores = {}
        for nbr in neighbors:
            edge_key = (current, nbr, 0)
            count = edge_counts.get(edge_key, 0)

            # Penalize already-walked edges heavily
            if edge_key in visited_edges:
                count += 50_000

            # Penalize recently visited nodes (tabu)
            if nbr in tabu:
                count += 20_000

            scores[nbr] = count

        min_score = min(scores.values())
        candidates = [n for n, s in scores.items() if s == min_score]
        next_node = random.choice(candidates)

        visited_edges.add((current, next_node, 0))
        tabu.append(current)
        if len(tabu) > tabu_length:
            tabu.pop(0)

        path.append(next_node)
        current = next_node

    return path
start_node = ox.nearest_nodes(G, START_LON, START_LAT)
path = least_used_route(G, edge_counts, start_node, steps=200)
print(f"Route has {len(path)} nodes")

In [ ]:
def most_used_route(G, edge_counts, start_node, steps=200):
    path = [start_node]
    current = start_node
    prev = None

    for _ in range(steps):
        neighbors = list(G.successors(current))
        if not neighbors:
            neighbors = list(G.neighbors(current))
        if not neighbors:
            break

        scores = {}
        for nbr in neighbors:
            count = edge_counts.get((current, nbr, 0), 0)
            if nbr == prev:
                count -= 10_000  # penalize backtracking
            scores[nbr] = count

        max_score = max(scores.values())
        candidates = [n for n, s in scores.items() if s == max_score]
        next_node = random.choice(candidates)
        prev = current
        path.append(next_node)
        current = next_node

    return path


start_node = ox.nearest_nodes(G, START_LON, START_LAT)
path_most = most_used_route(G, edge_counts, start_node, steps=100)
print(f"Route has {len(path_most)} nodes")

In [ ]:
import folium
from folium.plugins import PolyLineTextPath
import math

# Base map centered on city
center_lat = (LAT_MIN + LAT_MAX) / 2
center_lon = (LON_MIN + LON_MAX) / 2
m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="CartoDB positron")

# Draw all street edges colored by usage
max_count = max(edge_counts.values()) if edge_counts else 1
max_log = math.log1p(max_count) 
disregarded = 0
for (u, v, k), count in edge_counts.items(): 
    if not G.has_edge(u, v, k):
        disregarded +=1
        continue
    edge_data = G[u][v][k]
    if "geometry" in edge_data:
        coords = [(lat, lon) for lon, lat in edge_data["geometry"].coords]
    else:
        coords = [
            (G.nodes[u]["y"], G.nodes[u]["x"]),
            (G.nodes[v]["y"], G.nodes[v]["x"])
        ]
    intensity = math.log1p(count) / max_log  # 0.0 → 1.0
    weight  = 1 + intensity * 5              # 1px → 9px
    opacity = 0.3 + intensity * 0.7        # 0.15 → 1.0
    folium.PolyLine(coords, color="#ff00b3", weight=weight,
                    opacity=opacity).add_to(m)


print(disregarded, "edges were in counts but not graph")

route_coords_least = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in path]
route_coords_most  = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in path_most]

folium.PolyLine(route_coords_least, color="#424663", weight=3, opacity=0.9, tooltip="Least-used route").add_to(m)
#folium.PolyLine(route_coords_most,  color="#ff2200", weight=5, opacity=0.9, tooltip="Most-used route").add_to(m) #Uncomment if you want to show the most-used route. 


folium.Marker(route_coords_least[0], tooltip="Start",
              icon=folium.Icon(color="gray", icon="play")).add_to(m)
m.save("least_used_map.html")
from IPython.display import IFrame
IFrame("least_used_map.html", width=900, height=600)

m.save("least_used_map.html")
m  # Displays inline in Jupyter

In [ ]:
save_map_as_pdf("least_used_map.html", "least_used_map.pdf", wait=5)